In [1]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures','figures/figure2','figures/figure3','figures/figure4','figures/figure5'):
    _P(_d).mkdir(parents=True, exist_ok=True)

import sys, os
sys.path.insert(0, os.path.abspath("src"))
from alphaPhosHelperFunctions import *
import re
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
import analytics_core_V04 as ac
from core import *

In [2]:
# Cell-number gradients (matching Figure3_v00)
color_sequence_red    = ['#FBA08D', '#FA7A61', '#F95534', '#ED2E07', '#CB2706', '#9E1E05']
color_sequence_violet = ['#C79EEA', '#B178E2', '#9B52DA', '#7E2AC7', '#6D25AD', '#551D87']
CELL_ORDER = [100, 300, 500, 1000, 2000, 3000]

# Data upload

In [3]:
# Revision Figure 3 data = wide PTM Site Reports (per-run localization at 0.75),
# same format as Figure 2. Per-cell-number sorted datasets are bucketed by
# workflow (nanoPhos/uPhos) x system (HeLa/stem); the multi-condition stem-cell
# reports (2iL/SL/RA/RA24) are loaded separately.
RAW_DIR = Path('pride_data/analysis_data/revision/figure3')

def parse_filename(fname):
    workflow = 'nanophos' if 'nanoPhos' in fname else ('uphos' if 'uPhos' in fname else 'unknown')
    if 'HeLa' in fname:
        system = 'hela'
    elif 'StemCells' in fname:
        system = 'stem'
    else:
        system = 'unknown'
    m = re.search(r'(\d+)cells', fname)
    return workflow, system, (int(m.group(1)) if m else None)

buckets = {}
for path in sorted(RAW_DIR.glob('*.tsv')):
    if 'RAlin' in path.name or 'normalized' in path.name:
        continue   # multi-condition stem files handled below
    w, s, n = parse_filename(path.name)
    if n is None:
        continue
    buckets.setdefault(f'{w}_{s}', {})[n] = path

loaded = {}
for group, files in buckets.items():
    loaded[group] = {n: pd.read_csv(files[n], sep='\t', low_memory=False) for n in sorted(files)}
    print(f'{group:<16} cells: {sorted(files)}')

hela_nanophos = loaded.get('nanophos_hela', {})
hela_uphos    = loaded.get('uphos_hela',    {})
stem_nanophos = loaded.get('nanophos_stem', {})

# Multi-condition stem-cell datasets (2iL / SL / RA / RA24)
df_stem_all     = pd.read_csv(next(RAW_DIR.glob('*StemCells_RAlin_SL_all_Report.tsv')), sep='\t', low_memory=False)
df_stem_500norm = pd.read_csv(next(RAW_DIR.glob('*StemCells_RAlin_SL_500cells_normalized_Report.tsv')), sep='\t', low_memory=False)
print('df_stem_all:', df_stem_all.shape, '| df_stem_500norm:', df_stem_500norm.shape)

uphos_hela       cells: [100, 300, 500, 1000, 2000, 3000]
nanophos_hela    cells: [100, 300, 500, 1000, 2000, 3000]
nanophos_stem    cells: [100, 300, 500, 800, 1000, 3000]
df_stem_all: (41063, 152) | df_stem_500norm: (13493, 32)


# Figure 3a

In [6]:
# Figure 3a — HeLa-sorted nanoPhos phosphosite depth vs cell number.
# Class I, multiplicity collapsed, localization enforced (hardened counter). The site
# reports are already 0.75-filtered, so every counted site is Class I (no stacked
# "non-localized" fraction, unlike the v00 cumulative barplot). Box + points, n=3.
import importlib, core
importlib.reload(core)
from core import count_sites_per_sample_ptm_report, _hex_to_rgba
import numpy as np
import pandas as pd
import plotly.graph_objects as go

counts_by_cells = {n: list(count_sites_per_sample_ptm_report(hela_nanophos[n]).values())
                   for n in CELL_ORDER if n in hela_nanophos}

rows = []
for n in CELL_ORDER:
    if n not in counts_by_cells:
        continue
    c = counts_by_cells[n]
    rows.append({
        'cells':  n,
        'n_reps': len(c),
        'median': int(np.median(c)),
        'mean':   int(np.mean(c)),
        'std':    int(np.std(c, ddof=1)) if len(c) > 1 else 0,
        'CV%':    round(100 * np.std(c, ddof=1) / np.mean(c), 1) if len(c) > 1 else 0,
    })
summary = pd.DataFrame(rows).set_index('cells')
print(summary.to_string())

BOX_COLOR, POINT_COLOR = '#8A0000', '#393E46'
order = [n for n in CELL_ORDER if n in counts_by_cells]
fig = go.Figure()
for n in order:
    ys = counts_by_cells[n]
    x = str(n)
    fig.add_trace(go.Box(
        y=ys, x=[x] * len(ys), name=x,
        boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=9, color=POINT_COLOR, line=dict(width=0.5, color='black')),
        line=dict(color=BOX_COLOR, width=1.5),
        fillcolor=_hex_to_rgba(BOX_COLOR, 0.15),
        showlegend=False,
    ))
fig.update_layout(template='plotly_white', width=600, height=600,
                  xaxis_title='Sorted cells', yaxis_title='Class I phosphosites',
                  showlegend=False)
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in order])
fig.update_yaxes(range = [0, 18100])
fig.show()
fig.write_image(r'figures/figure3/figure3a.pdf', width=600, height=600)


       n_reps  median   mean  std   CV%
cells                                  
100         3    2559   2593  266  10.3
300         3    6721   6241  845  13.6
500         3    8807   8235  999  12.1
1000        3   10427  10410  433   4.2
2000        3   13750  13709  144   1.1
3000        3   15685  15627  129   0.8


# Figure 3b

In [8]:
# Figure 3b — per-site dilution linearity across sorted-cell inputs (HeLa nanoPhos).
# x = estimated protein input = (sorted cells) x 0.25 pg/cell. Per project policy,
# linearity KEEPS multiplicity (per-feature quantitative response); enforce_cutoff=0.75.
import importlib, core
importlib.reload(core)
from core import calculate_dilution_linearity
import numpy as np
import plotly.graph_objects as go

estimated_protein_input = [n * 0.25 for n in sorted(hela_nanophos)]   # pg (HeLa ~ 0.25 pg/cell)
corr_df = calculate_dilution_linearity(
    hela_nanophos, dilution_values=estimated_protein_input,
    min_dilutions=4, collapse_multiplicity=False, enforce_cutoff=0.75,
)
a = corr_df['r_squared'].dropna().values
print(f'n sites fit = {len(a):,}   median R^2 = {np.nanmedian(a):.4f}   '
      f'%(R^2 >= 0.8) = {100*(a >= 0.8).mean():.1f}%')

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=a, nbinsx=max(len(a) // 20, 1),
    marker=dict(color='black', line=dict(color='black', width=0.3)), opacity=0.8,
))
fig.add_vline(x=np.nanmedian(a), line={'dash': 'dash', 'width': 3, 'color': 'darkred'})
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='R squared', yaxis_title='Count',
                  font=dict(size=12), showlegend=False, bargap=0.05)
fig.show()
fig.write_image(r'figures/figure3/figure3b.pdf', height=600, width=600)


n sites fit = 9,468   median R^2 = 0.9320   %(R^2 >= 0.8) = 71.4%


In [9]:
np.nanmedian(a)

np.float64(0.9320033381279951)

# Figure 3c

In [11]:
# Figure 3c — nanoPhos vs uPhos depth advantage per sorted-cell number.
# Class I, multiplicity collapsed, localization enforced. Reported metric = ratio of
# per-condition MEANS; plotted points = each nanoPhos rep / uPhos mean (3 per cell
# number); black mean +/- SD crossbar. Same construction as the corrected Fig 2E.
import importlib, core
importlib.reload(core)
from core import count_sites_per_sample_ptm_report
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

summary_rows, ratios, labels = [], [], []
for n in CELL_ORDER:
    if n not in hela_nanophos or n not in hela_uphos:
        continue
    nano       = list(count_sites_per_sample_ptm_report(hela_nanophos[n]).values())
    uphos      = list(count_sites_per_sample_ptm_report(hela_uphos[n]).values())
    uphos_mean = np.mean(uphos)
    if uphos_mean <= 0:
        continue
    per_rep = [x / uphos_mean for x in nano]
    ratios += per_rep
    labels += [str(n)] * len(per_rep)
    summary_rows.append({
        'cells':          n,
        'nano_mean':      int(np.mean(nano)),
        'uphos_mean':     round(uphos_mean, 1),
        'ratio_of_means': round(np.mean(nano) / uphos_mean, 2),
    })

summary = pd.DataFrame(summary_rows).set_index('cells')
print(summary.to_string())

df_ratio = pd.DataFrame({'Ratio': ratios, 'ID': labels})
order = [str(n) for n in CELL_ORDER if n in hela_nanophos and n in hela_uphos]
fig = px.strip(df_ratio, y='Ratio', x='ID', log_y=True, category_orders={'ID': order})
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False,
                  xaxis_title='Sorted cells', yaxis_title='nanoPhos / uPhos Class I ratio')
fig.update_traces(marker=dict(size=18, color='#B3321E', line=dict(width=0.5, color='black')))
fig.update_yaxes(showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2', griddash='solid')
fig.update_xaxes(showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2', griddash='solid')
fig.add_hline(y=1.0, line=dict(color='black', dash='dot', width=1.5))

# mean +/- SD crossbar per cell number (n=3)
stats_c = df_ratio.groupby('ID')['Ratio'].agg(['mean', 'std']).reindex(order)
fig.add_trace(go.Scatter(
    x=stats_c.index, y=stats_c['mean'],
    error_y=dict(type='data', array=stats_c['std'].fillna(0), visible=True,
                 color='black', thickness=1.5, width=10),
    mode='markers',
    marker=dict(symbol='line-ew', size=26, color='black', line=dict(width=2, color='black')),
    showlegend=False, hovertemplate='mean=%{y:.1f}x<extra></extra>',
))
fig.show()
fig.write_image(r'figures/figure3/figure3c.pdf', width=600, height=600)


       nano_mean  uphos_mean  ratio_of_means
cells                                       
100         2593       501.3            5.17
300         6241      2073.3            3.01
500         8235      3178.3            2.59
1000       10410      3955.0            2.63
2000       13709      5865.0            2.34
3000       15627      6001.3            2.60


# Figure 3e

In [14]:
# Figure 3e — stem-cell phosphoproteome PCA (2iL / SL / RA across cell numbers).
# No-cross-run-normalization reanalysis of StemCells_RAlin_SL_all (Class I 0.75), same
# pipeline as Figure 2F. cellline->state mapping VERIFIED from the Figure3_v00 condition_df
# sample order: cellline1=2iL, 2=SL, 4=RA, 5=RA24 (RA24 excluded from the PCA, as in v00).
# Note: ac.set_condition matches sample IDs by exact '_'-split token, so renamed IDs are
# underscore-free; the group key keeps the underscore (state_cellnumber).
import importlib, core
importlib.reload(core)
from core import process_ptm_site_report
import re
from collections import defaultdict
import pandas as pd
import plotly.express as px

CELLLINE_TO_STATE = {1: '2iL', 2: 'SL', 4: 'RA', 5: 'RA24'}
STEM_CELL_ORDER   = [100, 300, 500, 800, 1000, 3000]

df_stem_wonorm = pd.read_csv(
    next(RAW_DIR.glob('*StemCells_RAlin_SL_all_wo_norm_Report.tsv')), sep='\t', low_memory=False)
site_data_all = process_ptm_site_report(df_stem_wonorm, cutoff=0.75)['site_data']

PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}
sample_cols = [c for c in site_data_all.columns if c not in PROC_META]

def parse_stem(name):
    cl = re.search(r'cellline(\d+)', name)
    nc = re.search(r'(\d+)cells', name)
    if not cl or not nc:
        return None
    state = CELLLINE_TO_STATE.get(int(cl.group(1)))
    return None if state is None else f'{state}_{int(nc.group(1))}'

condition_to_samples = defaultdict(list)
for s in sample_cols:
    cond = parse_stem(s)
    if cond:
        condition_to_samples[cond].append(s)

rename_map, dict_cond_all = {}, {}
for cond, samples in condition_to_samples.items():
    base = cond.replace('_', '')
    ids = []
    for i, s in enumerate(sorted(samples)):
        sid = f'{base}c{i+1:02d}'
        rename_map[s] = sid
        ids.append(sid)
    dict_cond_all[cond] = ids

site_data_renamed = site_data_all.rename(columns=rename_map)
AC_META_REQUIRED = {'UPD_seq', 'PTM_localization', 'Protein_group', 'Gene_group', 'PTM_Collapse_key'}
site_data_for_ac = site_data_renamed.drop(
    columns=[c for c in PROC_META if c not in AC_META_REQUIRED and c in site_data_renamed.columns])

site_data_grouped = ac.set_condition(site_data_for_ac, dict_cond_all)
site_data_filt    = ac.filt_per_percentage(site_data_grouped, 0.7)
site_data_imp     = ac.imputation_normal_distribution(site_data_filt).reset_index()
site_data_imp = site_data_imp[~site_data_imp['group'].str.contains('RA24')]
pca = ac.run_pca(site_data_imp)
print('Groups:', site_data_imp['group'].nunique(), '| PCA explained variance:', pca[1])


Dropped 7,011 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 513,173 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 34,053 → 33,953.
Final: 33,953 sites × 72 samples.


d:\Projects\nanoPhos_env\src\analytics_core_V04.py:308: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Groups: 18 | PCA explained variance: {'x_title': 'PC1 (0.60)', 'y_title': 'PC2 (0.05)', 'group': 'group'}


In [18]:
# Figure 3e plot — colored by state (RA=blue, 2iL=red, SL=violet), shaded by cell number.
# Conservative reading (Reviewer 1): RA separates from SL/2iL on PC1; SL and 2iL overlap;
# PC1 is also partly driven by cell number. Avoid 'clear separation of all three states'.
pca_df = pca[0][0]
blue_shades = ['#9ECCEA', '#78B8E2', '#52A4DA', '#2A88C7', '#2576AD', '#1D5C87']
STATE_PALETTE = {'RA': blue_shades, '2iL': color_sequence_red, 'SL': color_sequence_violet}

color_map = {}
for grp in pca_df['group'].unique():
    state, ncell = grp.rsplit('_', 1)
    color_map[grp] = STATE_PALETTE[state][STEM_CELL_ORDER.index(int(ncell))]

fig = px.scatter(pca_df, x='x', y='y', color='group', color_discrete_map=color_map)
fig.update_layout(width=600, height=600, template='plotly_white', showlegend = False)
fig.update_traces(marker=dict(size=21, line=dict(width=1, color='black')))
fig.update_xaxes(title=pca[1]['x_title'])
fig.update_yaxes(title=pca[1]['y_title'])
fig.show()
fig.write_image(r'figures/figure3/figure3e.pdf', height=600, width=600)


# Figure 3F

In [20]:
# Figure 3F — phosphosite depth per pluripotency state (500-cell normalized stem experiment).
# Per-sample Class I counts grouped by state (cellline1=2iL, 2=SL, 4=RA, 5=RA24).
# RA24 excluded for consistency with 3e; set STATE_ORDER to include 'RA24' to add it back.
# Colors match 3e: 2iL=red, SL=violet, RA=blue.
import importlib, core
importlib.reload(core)
from core import count_sites_per_sample_ptm_report, _hex_to_rgba
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go

CELLLINE_TO_STATE = {1: '2iL', 2: 'SL', 4: 'RA', 5: 'RA24'}
STATE_ORDER = ['2iL', 'SL', 'RA']                                   # RA24 excluded (as in 3e)
STATE_COLOR = {'2iL': '#ED2E07', 'SL': '#7E2AC7', 'RA': '#2A88C7'}  # red / violet / blue

counts = count_sites_per_sample_ptm_report(df_stem_500norm)
by_state = {s: [] for s in STATE_ORDER}
for sample, n in counts.items():
    m = re.search(r'cellline(\d+)', sample)
    state = CELLLINE_TO_STATE.get(int(m.group(1))) if m else None
    if state in by_state:
        by_state[state].append(n)

for s in STATE_ORDER:
    c = by_state[s]
    print(f'{s:<4} n={len(c)}  mean={int(np.mean(c))}  CV%={round(100*np.std(c, ddof=1)/np.mean(c),1)}  vals={c}')

fig = go.Figure()
for s in STATE_ORDER:
    ys = by_state[s]
    fig.add_trace(go.Box(
        y=ys, x=[s] * len(ys), name=s,
        boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=9, color=STATE_COLOR[s], line=dict(width=0.5, color='black')),
        line=dict(color=STATE_COLOR[s], width=1.5),
        fillcolor=_hex_to_rgba(STATE_COLOR[s], 0.2),
        showlegend=False,
    ))
fig.update_layout(width=500, height=600, template='plotly_white', showlegend=False,
                  xaxis_title='Pluripotency state (500 cells)', yaxis_title='Class I phosphosites')
fig.update_xaxes(categoryorder='array', categoryarray=STATE_ORDER)
fig.update_yaxes(rangemode='tozero')
fig.show()
fig.write_image(r'figures/figure3/figure3f.pdf', width=600, height=600)


2iL  n=3  mean=4387  CV%=6.6  vals=[4668, 4089, 4405]
SL   n=3  mean=4265  CV%=5.6  vals=[4328, 4465, 4002]
RA   n=3  mean=4807  CV%=2.0  vals=[4886, 4698, 4838]


# Figure 3G

In [22]:
# Figure 3G — per-sample phosphosite intensity distributions (500-cell normalized stem).
# log2 intensities from process_ptm_site_report; one horizontal violin per sample,
# colored by state (2iL=red, SL=violet, RA=blue). Shows 2iL/SL/RA (v00 plotted RA24 for
# the third group via positional slicing — corrected here to RA, consistent with 3e/3F).
import importlib, core
importlib.reload(core)
from core import process_ptm_site_report
import re
import numpy as np
import plotly.graph_objects as go

CELLLINE_TO_STATE = {1: '2iL', 2: 'SL', 4: 'RA', 5: 'RA24'}
STATE_ORDER = ['2iL', 'SL', 'RA']
STATE_COLOR = {'2iL': '#e93526', 'SL': '#734b9e', 'RA': '#2b89c7'}   # red / violet / blue

sd = process_ptm_site_report(df_stem_500norm, cutoff=0.75)['site_data']
PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}
sample_cols = [c for c in sd.columns if c not in PROC_META]

def state_of(col):
    m = re.search(r'cellline(\d+)', col)
    return CELLLINE_TO_STATE.get(int(m.group(1))) if m else None

# one row per sample, grouped by state (2iL, SL, RA), top-to-bottom
rows = []
for s in STATE_ORDER:
    for j, col in enumerate(sorted(c for c in sample_cols if state_of(c) == s), 1):
        rows.append((f'{s}{j}', s, col))

fig = go.Figure()
for label, s, col in rows:
    fig.add_trace(go.Violin(
        x=sd[col].dropna().tolist(), y=[label] * sd[col].notna().sum(),
        line_color=STATE_COLOR[s], fillcolor=STATE_COLOR[s], opacity=0.6,
        orientation='h', side='positive', width=3, points=False, showlegend=False,
    ))
overall_med = float(np.nanmedian(np.concatenate([sd[c].dropna().values for _, _, c in rows])))
fig.add_vline(x=overall_med, line_dash='dash', line_color='darkslategrey', line_width=2, opacity=0.7)
fig.update_layout(width=600, height=500, template='plotly_white', showlegend=False,
                  xaxis_title='log2 phosphosite intensity',
                  yaxis=dict(categoryorder='array', categoryarray=[r[0] for r in rows]))
fig.show()
fig.write_image(r'figures/figure3/figure3g.pdf', width=600, height=500)


Dropped 2,230 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 57,575 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 11,263 → 11,232.
Final: 11,232 sites × 12 samples.


# Figure 3H

In [40]:
# Figure 3H — marker-gene phosphosite clustering (500-cell normalized).
# Pipeline: strict per-run Class I  ->  valid-value filter (>=67% real values in EACH of
# 2iL/SL/RA, i.e. >=2/3 reps)  ->  down-shifted imputation  ->  one-way ANOVA (FDR<0.05).
# The heatmap shows ONLY ANOVA-significant marker-gene phosphosites (no whole-condition
# imputation drives significance, because every tested site has real data in all 3 states).
import importlib, core
importlib.reload(core)
from core import process_ptm_site_report
import re
import numpy as np
import pandas as pd
import analytics_core_V04 as ac

CELLLINE_TO_STATE = {1: '2iL', 2: 'SL', 4: 'RA', 5: 'RA24'}
STATE_ORDER  = ['2iL', 'SL', 'RA']
MARKER_GENES = ['Pou3f1', 'Dnmt3a', 'Dbn1', 'Krt18', 'Sox3', 'Esrrb', 'Sox2', 'Otx2',
                'Dnmt3b', 'Pou5f1', 'Dnmt3l', 'Tet2', 'Nanog', 'Nes']

df0 = pd.read_csv(next(RAW_DIR.glob('*StemCells_RAlin_SL_500cells_normalized_0_cutoff_Report.tsv')),
                  sep='\t', low_memory=False)
sd = process_ptm_site_report(df0, cutoff=0.75)['site_data']                  # strict per-run Class I
PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}
def state_of(c):
    m = re.search(r'cellline(\d+)', c)
    return CELLLINE_TO_STATE.get(int(m.group(1))) if m else None
cmap = {s: sorted(c for c in sd.columns if c not in PROC_META and state_of(c) == s) for s in STATE_ORDER}
sample_cols = [c for s in STATE_ORDER for c in cmap[s]]

# valid-value filter: >=67% real Class I values in EACH condition (>=2 of 3 reps)
ok = pd.Series(True, index=sd.index)
for s in STATE_ORDER:
    ok &= sd[cmap[s]].notna().sum(axis=1) >= 2

rename, dca = {}, {}
for s in STATE_ORDER:
    ids = []
    for i, c in enumerate(cmap[s]):
        sid = f'{s}c{i+1:02d}'; rename[c] = sid; ids.append(sid)
    dca[s] = ids
AC_REQ = {'UPD_seq', 'PTM_localization', 'Protein_group', 'Gene_group', 'PTM_Collapse_key'}
sfa = sd[ok][[c for c in sd.columns if c in sample_cols or c in AC_REQ]].rename(columns=rename)
si  = ac.imputation_normal_distribution(ac.set_condition(sfa, dca)).reset_index()   # down-shifted impute
anova = ac.run_anova(si)                                                            # one-way ANOVA + FDR
sig_keys = set(anova.loc[anova['rejected'] == True, 'identifier'])

# significant marker matrix (sites x samples, imputed -> complete)
site_cols = [c for c in si.columns if c not in ('group', 'sample', 'subject', 'index')]
mat = si.set_index('sample')[site_cols].T
def gene_of(k):
    try:
        return k.split('~')[1].split('_')[0]
    except Exception:
        return None
mat = mat[[(k in sig_keys) for k in mat.index]]                 # ANOVA-significant only
mat = mat[[gene_of(k) in MARKER_GENES for k in mat.index]]      # marker genes only
print(f'{len(mat)} ANOVA-significant marker phosphosites; genes:',
      sorted({gene_of(k) for k in mat.index}))


Dropped 2,420 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 141,058 of 201,346 intensity cells masked (70.1%).
Dropped 8,178 sites with no remaining quant after masking.
Deduplicated multi-protein rows: 11,546 → 11,514.
Final: 11,514 sites × 12 samples.


d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Pro

8 ANOVA-significant marker phosphosites; genes: ['Dnmt3a', 'Dnmt3b', 'Dnmt3l', 'Esrrb', 'Nes', 'Sox2']


d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.

d:\Projects\nanoPhos_env\Lib\site-packages\pingouin\pairwise.py:28: UserWarning:

pairwise_ttests is deprecated, use pairwise_tests instead.



In [41]:
# Figure 3H plot - Plotly clustered heatmap (row + column dendrograms, z-scored).
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff
from scipy.cluster.hierarchy import linkage
from sklearn.preprocessing import StandardScaler

# readable GENE_pSite row labels; multiplicity made explicit so same-site M1/M2
# features (e.g. Dnmt3l pS61 mono- vs di-phosphorylated peptide) are distinguishable.
def short_label(k):
    parts = k.split('~')[1].split('_')
    gene, site = parts[0], parts[1]
    mult = parts[2] if len(parts) > 2 else ''
    return f'{gene}_p{site}' + (f' ({mult})' if mult else '')

def uniquify(labels):
    seen, out = {}, []
    for l in labels:
        if l in seen:
            seen[l] += 1; out.append(f'{l}.{seen[l]}')
        else:
            seen[l] = 0; out.append(l)
    return out

df_heat = mat.copy()
df_heat.index = uniquify([short_label(k) for k in df_heat.index])

# z-normalize per site (row)
df_norm = pd.DataFrame(StandardScaler().fit_transform(df_heat.T).T,
                       index=df_heat.index, columns=df_heat.columns)

linkfun = lambda x: linkage(x, method='ward', metric='euclidean')
X = df_norm.values
row_labels, col_labels = df_norm.index.tolist(), df_norm.columns.tolist()

# top (column / sample) dendrogram
fig = ff.create_dendrogram(X.T, orientation='bottom', labels=col_labels, linkagefun=linkfun)
for d in fig['data']:
    d['yaxis'] = 'y2'

# left (row / site) dendrogram
dside = ff.create_dendrogram(X, orientation='right', labels=row_labels, linkagefun=linkfun)
for d in dside['data']:
    d['xaxis'] = 'x2'
for d in dside['data']:
    fig.add_trace(d)

# reorder matrix to dendrogram leaf order
col_order = list(fig['layout']['xaxis']['ticktext'])
row_order = list(dside['layout']['yaxis']['ticktext'])
z = df_norm.loc[row_order, col_order].values

fig.add_trace(go.Heatmap(
    x=fig['layout']['xaxis']['tickvals'],
    y=dside['layout']['yaxis']['tickvals'],
    z=z, colorscale='RdBu_r', zmid=0,
    colorbar={'title': 'Z-score', 'len': 0.7, 'y': 0.43, 'yanchor': 'middle'},
))

# layout: heatmap + two dendrogram panels
fig.update_layout(
    width=820, height=max(520, 22 * len(row_order) + 160),
    showlegend=False, template='simple_white', hovermode='closest',
    margin=dict(l=10, r=10, t=20, b=120),
)
# heatmap region (bottom-right)
fig.update_layout(xaxis={'domain': [0.16, 1.0], 'showgrid': False, 'showline': False,
                         'zeroline': False, 'ticks': '', 'tickangle': 45,
                         'ticktext': col_order, 'tickvals': list(fig['layout']['xaxis']['tickvals'])})
fig.update_layout(yaxis={'domain': [0.0, 0.85], 'showgrid': False, 'showline': False,
                         'zeroline': False, 'side': 'right', 'ticks': '',
                         'ticktext': row_order, 'tickvals': list(dside['layout']['yaxis']['tickvals'])})
# top dendrogram region
fig.update_layout(yaxis2={'domain': [0.86, 1.0], 'showgrid': False, 'showline': False,
                          'zeroline': False, 'showticklabels': False, 'ticks': ''})
# left dendrogram region
fig.update_layout(xaxis2={'domain': [0.0, 0.15], 'showgrid': False, 'showline': False,
                          'zeroline': False, 'showticklabels': False, 'ticks': ''})

fig.show()
# fig.write_html(r'figures/figure3/figure3h.html')
# fig.write_image(r'figures/figure3/figure3h.pdf', scale=2)  # needs kaleido


In [43]:
# [DISABLED for public repo — PRIDE MetaInfo export; uncomment to regenerate MetaInfo v02]
# # === PRIDE MetaInfo export -- Figure 3 panels (run after all panels above) ===
# import sys; sys.path.insert(0, r"src")
# from metainfo_export import dump_panel
# from core import count_sites_per_sample_ptm_report, process_ptm_site_report
# import re, pandas as pd
# _META = {'Protein_group','Gene_group','PTM_0_aa','PTM_pos','PTM_mult123','PTM_flank','PTM_Collapse_key','PTM_localization','UPD_seq'}
# def _try(fn, sheet):
#     try: fn()
#     except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")
# def _3a():
#     _r=[]
#     for n in sorted(hela_nanophos):
#         for rep,(samp,c) in enumerate(count_sites_per_sample_ptm_report(hela_nanophos[n]).items(),1):
#             _r.append({'Raw file':samp,'Condition':f'{n}cells','Replicate':rep,'Number of class I sites':int(c)})
#     dump_panel(pd.DataFrame(_r), "Figure 3a")
# _try(_3a,"Figure 3a")
# _try(lambda: dump_panel(corr_df, "Figure 3b"), "Figure 3b")
# _try(lambda: dump_panel(df_ratio, "Figure 3c"), "Figure 3c")
# _try(lambda: dump_panel(pca_df.rename(columns={'x':'PC1','y':'PC2'}), "Figure 3e"), "Figure 3e")
# def _3f():
#     _CL={1:'2iL',2:'SL',4:'RA',5:'RA24'}; _r=[]
#     for samp,c in count_sites_per_sample_ptm_report(df_stem_500norm).items():
#         m=re.search(r'cellline(\d+)',samp); st=_CL.get(int(m.group(1))) if m else None
#         _r.append({'Raw file':samp,'State':st,'Number of class I sites':int(c)})
#     dump_panel(pd.DataFrame(_r), "Figure 3f")
# _try(_3f,"Figure 3f")
# def _3g():
#     _sd=process_ptm_site_report(df_stem_500norm,cutoff=0.75)['site_data']
#     dump_panel(_sd[['PTM_Collapse_key']+[c for c in _sd.columns if c not in _META]], "Figure 3g")
# _try(_3g,"Figure 3g")
# _try(lambda: dump_panel(df_norm.reset_index().rename(columns={'index':'site'}), "Figure 3h"), "Figure 3h")
# print("Figure 3 panels exported to MetaInfo v02.")


  [MetaInfo] wrote 'Figure 3a'  (18 rows x 4 cols)
  [MetaInfo] wrote 'Figure 3b'  (9468 rows x 7 cols)
  [MetaInfo] wrote 'Figure 3c'  (18 rows x 2 cols)
  [MetaInfo] wrote 'Figure 3e'  (54 rows x 4 cols)
  [MetaInfo] wrote 'Figure 3f'  (12 rows x 3 cols)
Dropped 2,230 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 57,575 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 11,263 → 11,232.
Final: 11,232 sites × 12 samples.
  [MetaInfo] wrote 'Figure 3g'  (11232 rows x 13 cols)
  [MetaInfo] wrote 'Figure 3h'  (8 rows x 10 cols)
Figure 3 panels exported to MetaInfo v02.
